<a href="https://colab.research.google.com/github/shin-noda/leetcode-problemset-97/blob/main/Problem1044_Improved2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
class SAIS:
    def is_lms(self, i, t):
        return i > 0 and t[i] and not t[i - 1]


    def get_buckets(self, alphabet_size, buckets):
        head = [0] * alphabet_size
        tail = [0] * alphabet_size
        acc = 0

        for i, c in enumerate(buckets):
            head[i] = acc
            acc += c
            tail[i] = acc

        return head, tail


    def induce(self, lms, A, t, alphabet_size, buckets):
        n = len(A)
        sa = [-1] * n

        head, tail = self.get_buckets(alphabet_size, buckets)
        for idx in reversed(lms):
            c = A[idx]
            tail[c] -= 1
            sa[tail[c]] = idx

        head, tail = self.get_buckets(alphabet_size, buckets)
        for i in range(n):
            if sa[i] > 0 and not t[sa[i] - 1]:
                c = A[sa[i] - 1]
                sa[head[c]] = sa[i] - 1
                head[c] += 1

        head, tail = self.get_buckets(alphabet_size, buckets)
        for i in range(n - 1, -1, -1):
            if sa[i] > 0 and t[sa[i] - 1]:
                c = A[sa[i] - 1]
                tail[c] -= 1
                sa[tail[c]] = sa[i] - 1

        return sa


    def lms_equal(self, A, t, a, b):
        i = 0

        while True:
            a_end = self.is_lms(a + i, t)
            b_end = self.is_lms(b + i, t)

            if A[a + i] != A[b + i] or t[a + i] != t[b + i]:
                return False

            if i > 0 and a_end and b_end:
                return True

            if a_end != b_end:
                return False

            i += 1


    def process(self, A, alphabet_size):
        n = len(A)

        if n <= 1:
            return list(range(n))

        t = [False] * n
        t[-1] = True
        for i in range(n - 2, -1, -1):
            if A[i] < A[i + 1]:
                t[i] = True

            elif A[i] > A[i + 1]:
                t[i] = False

            else:
                t[i] = t[i + 1]

        buckets = [0] * alphabet_size
        for x in A:
            buckets[x] += 1

        lms = [i for i in range(n) if self.is_lms(i, t)]
        sa = self.induce(lms, A, t, alphabet_size, buckets)
        sorted_lms = [x for x in sa if self.is_lms(x, t)]
        lms_id = [-1] * n
        name = 0
        lms_id[sorted_lms[0]] = 0

        for i in range(1, len(sorted_lms)):
            u = sorted_lms[i - 1]
            v = sorted_lms[i]

            if not self.lms_equal(A, t, u, v):
                name += 1

            lms_id[v] = name

        summary = [lms_id[i] for i in lms]

        if name + 1 < len(summary):
            summary_sa = self.process(summary, name + 1)

        else:
            summary_sa = list(range(len(summary)))

        ordered_lms = [lms[i] for i in summary_sa]

        return self.induce(ordered_lms, A, t, alphabet_size, buckets)


class Solution:
    def longestDupSubstring(self, s):
        # Convert string to integers (plus a 0 terminator for the '$' concept)
        # This makes it easy to handle recursion with renamed alphabet characters.
        A = [ord(ch) - ord('a') + 1 for ch in s] + [0]

        # Suffix Array:
        # It's an array of starting indices, sorted according to the lexicographic order
        # of the suffixes starting at those indices.
        # A suffix array is really a permutation of the indices 0...n - 1.
        # Nothing is created or destroyed.
        #
        # Run SA-IS to get the full Suffix Array
        # Note: # We don't hardcode 27.
        # alphabet_size is simply the number of distinct integer symbols
        # in the current string. During recursion, the alphabet becomes
        # the set of summary labels (0,1,2,...), not necessarily letters.

        sais = SAIS()
        suffix_array = sais.process(A, max(A) + 1)

        # The first element of suffix_array is always the virtual terminator we added
        # so we slice it out to match the original string length
        suffix_array = suffix_array[1:]

        # Build Longest Common Prefix (LCP) array using Kasai's Algorithm (O(N))
        n = len(s)
        lcp = [0] * n
        rank = [0] * n
        for i, sa_val in enumerate(suffix_array):
            rank[sa_val] = i

        h = 0
        max_len = 0
        start_idx = 0

        for i in range(n):
            if rank[i] > 0:
                j = suffix_array[rank[i] - 1]
                while i + h < n and j + h < n and s[i + h] == s[j + h]:
                    h += 1

            lcp[rank[i]] = h

            if h > max_len:
                max_len = h
                start_idx = i

            if h > 0:
                h -= 1

        if max_len <= 0:
            return ""

        return s[start_idx : start_idx + max_len]

In [27]:
solution = Solution()

In [28]:
s = "banana"

print(solution.longestDupSubstring(s))

ana


In [29]:
s = "abcd"

print(solution.longestDupSubstring(s))

In [30]:
s = "abcabxabcy"

print(solution.longestDupSubstring(s))

abc


In [31]:
s = "abaxab"

print(solution.longestDupSubstring(s))

ab


In [32]:
s = "abaxaeaf"

print(solution.longestDupSubstring(s))

a


In [33]:
s = "xabaaabg"

print(solution.longestDupSubstring(s))

aa
